In [1]:
from astropy.io import ascii, fits
from astropy.table import Table
TAB = Table(ascii.read('test_for_sed.csv'))
ROW = TAB[0]

from prospect.io import read_results as pread
from run_prosp_nonparaSFH import build_model, build_sps
results, observations_dict, _ = pread.results_from('test0_fit.h5')
model = build_model(ROW)

building model


In [2]:
# pread.subcorner(results)


In [3]:
from prospect.sources import CSPSpecBasis
def build_sps(**kwargs):
    """
    This is our stellar population model which generates the spectra for stars of a given age and mass. 
    Most of the time, you aren't going to need to pay attention to this. 
    """
    sps = CSPSpecBasis(zcontinuous=1)
    return sps

sps = build_sps()

In [4]:
from tqdm.auto import tqdm
import numpy as np
n_samples = 1000
n_wavelengths = sps.wavelengths.shape[0]
n_mags = len(observations_dict['filters'])
seds_spec_array = np.zeros((n_samples, n_wavelengths))
seds_mag_array = np.zeros((n_samples, n_mags))
surviving_mass_frac = np.zeros(n_samples)
weights = results.get('weights', None)
if weights is not None:
    idx = np.argsort(weights)[-n_samples:]
else:
    idx = np.arange(-n_samples, 0)

for i, chain_index in enumerate(tqdm(idx)):
    thetas = results['chain'][chain_index]
    spec, mags, mass_frac = model.predict(thetas, sps=sps,obs=observations_dict)
    seds_spec_array[i, :] = spec
    seds_mag_array[i, :] = mags
    surviving_mass_frac[i] = mass_frac


/Users/thardy/Documents/durham/eelgs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [ ]:
from corner import quantile
sed_dist = []
for i in tqdm(range(len(sps.wavelengths))):
    sed_dist.append(quantile([item[i] for item in seds_spec], [0.16, 0.5, 0.84]))